In [42]:
import pandas as pd
from pathlib import Path

from genrec.paths import ML1M_DIR

# only define folder if it was not already created in another cell
if "folder" not in globals():
    folder = ML1M_DIR   # data/ml-1m (genrec.paths)

schemas = {
    "ratings.dat": ["UserID", "MovieID", "Rating", "Timestamp"],
    "users.dat": ["UserID", "Gender", "Age", "Occupation", "ZipCode"],
    "movies.dat": ["MovieID", "Title", "Genres"],
}

dataframes = {}

for filename, columns in schemas.items():
    path = folder / filename
    if path.exists():
        dataframes[path.stem] = pd.read_csv(
            path,
            sep="::",
            engine="python",
            names=columns,
            encoding="latin-1",
        )
        print(f"{path.name}: {dataframes[path.stem].shape}")

# Access the loaded dataframes
ratings_df = dataframes.get("ratings")
movies_df = dataframes.get("movies")
users_df = dataframes.get("users")

ratings.dat: (1000209, 4)
users.dat: (6040, 5)
movies.dat: (3883, 3)


In [46]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jrobischon/wikipedia-movie-plots")

print("Path to dataset files:", path)

Path to dataset files: ~/.cache/kagglehub/datasets/jrobischon/wikipedia-movie-plots/versions/1


In [47]:
import os
import pandas as pd

wiki = pd.read_csv(os.path.join(path, "wiki_movie_plots_deduped.csv"))

print(wiki.shape)
wiki.head()


(34886, 8)


,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",American,Unknown,NaN,unknown,"https://en.wikipedia.org/wiki/Terrible_Teddy,_...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,American,"George S. Fleming, Edwin S. Porter",NaN,unknown,https://en.wikipedia.org/wiki/Jack_and_the_Bea...,The earliest known adaptation of the classic f...


In [48]:
import re
import unicodedata
from rapidfuzz import fuzz, process

ARTICLES = r"The|A|An|La|Le|Les|L'|Il|El|Das|Der|Die"
NUMBERS = {"ii": "2", "iii": "3", "iv": "4", "v": "5", "vi": "6", "vii": "7", "viii": "8",
           "one": "1", "two": "2", "three": "3", "four": "4", "five": "5"}

def normalize(title):
    """'Naked Gun 2 1/2: The Smell of Fear, The' -> 'naked gun 2 1 2 smell of fear'"""
    t = unicodedata.normalize("NFKD", str(title)).encode("ascii", "ignore").decode()
    t = re.sub(rf"^(.*), ({ARTICLES})$", r"\2 \1", t.strip())   # "Matrix, The" -> "The Matrix"
    t = t.lower().replace("&", " and ").replace("+", " and ")
    t = re.sub(r"[^a-z0-9 ]", " ", t)
    words = [NUMBERS.get(w, w) for w in t.split() if w != "part"]
    if words and words[0] in {"the", "a", "an"}:
        words = words[1:]
    return " ".join(words)

def split_names(title):
    """'Mad Max 2 (a.k.a. The Road Warrior)' -> ['Mad Max 2', 'The Road Warrior']"""
    alts = re.findall(r"\(([^()]*)\)", title)
    main = re.sub(r"\s*\([^()]*\)", "", title).strip()
    return [main] + [re.sub(r"^(a\.?k\.?a\.?)\s*", "", a, flags=re.I) for a in alts]

def title_keys(title):
    """One row per name: (key, base, is_main). base = part before ':' / ' - ' (for subtitle matches).
    The subtitle itself is added as an alternate name: 'Star Wars: Episode V - The Empire Strikes Back'."""
    names = split_names(title)
    main_parts = re.split(r":| - ", names[0])
    if len(main_parts) > 1 and len(main_parts[-1].split()) >= 3:
        names.append(main_parts[-1])
    rows = []
    for i, name in enumerate(names):
        base = re.split(r":| - ", name)[0]
        rows.append((normalize(name), normalize(base), i == 0))
    return [r for r in rows if r[0]]

In [49]:
# MovieLens side: "Toy Story (1995)" -> Year 1995 + title keys (movies_df itself is left untouched)
ml = movies_df[["MovieID", "Title"]].copy()
parsed = ml["Title"].str.extract(r"^(?P<Name>.*?)\s*\((?P<Year>\d{4})\)\s*$")
ml["Year"] = parsed["Year"].astype(int)
ml["Keys"] = parsed["Name"].map(title_keys)

# Wiki side: one row per (movie, name), grouped by release year for fast candidate lookup
wk = wiki[["Title", "Release Year", "Origin/Ethnicity"]].copy()
wk["wiki_idx"] = wiki.index
wk["Keys"] = wk["Title"].map(title_keys)
wk = wk.explode("Keys").dropna(subset=["Keys"])
wk[["key", "base", "is_main"]] = pd.DataFrame(wk.pop("Keys").tolist(), index=wk.index)
wk["compact"] = wk["key"].str.replace(" ", "")
wk_by_year = {y: g.reset_index(drop=True) for y, g in wk.groupby("Release Year")}

In [50]:
METHODS = ["exact", "fuzzy", "subtitle"]   # in order of trust

def match_movie(keys, year, fuzzy_cutoff=90):
    """Best wiki row for one MovieLens movie -> (wiki_idx, score, method, year_diff) or None.
    exact    : same normalized title (ignoring spaces), release year within +-1 (same year for alternate titles)
    fuzzy    : token_sort_ratio >= fuzzy_cutoff, same year     ('U.S. Marshalls' ~ 'U.S. Marshals')
    subtitle : one title is the other minus its subtitle, same year  ('Godzilla 2000' ~ 'Godzilla 2000: Millennium')
    """
    near = [wk_by_year[y] for y in (year - 1, year, year + 1) if y in wk_by_year]
    if not near:
        return None
    near = pd.concat(near, ignore_index=True)
    same = near[near["Release Year"] == year].reset_index(drop=True)
    found = []
    for key, base, is_main in keys:
        for _, c in near[near["compact"] == key.replace(" ", "")].iterrows():
            dy = abs(c["Release Year"] - year)
            if dy == 0 or (is_main and c["is_main"]):
                found.append(("exact", 100.0, dy, c))
        if len(same):
            for _, score, pos in process.extract(key, same["key"], scorer=fuzz.token_sort_ratio,
                                                 score_cutoff=fuzzy_cutoff, limit=3):
                found.append(("fuzzy", score, 0, same.iloc[pos]))
            sub = same[((same["base"] == key) & (same["key"] != key)) |
                       ((same["key"] == base) & (base != key))]
            for _, c in sub.iterrows():
                found.append(("subtitle", 100.0, 0, c))
    if not found:
        return None
    method, score, dy, c = min(found, key=lambda f: (METHODS.index(f[0]), -f[1], f[2],
                                                     f[3]["Origin/Ethnicity"] != "American"))
    return c["wiki_idx"], score, method, dy

res = [match_movie(k, y) for k, y in zip(ml["Keys"], ml["Year"])]
matches = pd.DataFrame([r if r else (None, None, None, None) for r in res],
                       columns=["wiki_idx", "match_score", "match_method", "year_diff"], index=ml.index)
ml = pd.concat([ml.drop(columns=matches.columns, errors="ignore"), matches], axis=1)

# a wiki row can only belong to one movie: keep the most trustworthy claim
ml["_rank"] = ml["match_method"].map({m: i for i, m in enumerate(METHODS)})
ml = ml.sort_values(["_rank", "match_score", "year_diff"], ascending=[True, False, True])
dup = ml["wiki_idx"].notna() & ml.duplicated("wiki_idx")
print("dropped duplicate claims:", ml.loc[dup, "Title"].tolist())
ml.loc[dup, ["wiki_idx", "match_score", "match_method", "year_diff"]] = None
ml = ml.drop(columns="_rank").sort_index()

dropped duplicate claims: ['Fear, The (1995)', 'Boys, The (1997)']


In [51]:
movies = movies_df.merge(
    ml[["MovieID", "Year", "wiki_idx", "match_score", "match_method"]], on="MovieID", how="left"
).merge(
    wiki.rename(columns={"Title": "Wiki Title", "Genre": "Wiki Genre"}),
    left_on="wiki_idx", right_index=True, how="left",
).drop(columns="wiki_idx")

print(movies.shape)
print(f"matched {movies['Plot'].notna().sum()} / {len(movies)} ({movies['Plot'].notna().mean():.1%})")
movies["match_method"].value_counts()

(3883, 14)
matched 2816 / 3883 (72.5%)


match_method
exact       2801
fuzzy          8
subtitle       7
Name: count, dtype: int64

In [52]:
# review the non-exact matches
movies[movies["match_method"].isin(["fuzzy", "subtitle"])][
    ["Title", "Wiki Title", "Release Year", "Origin/Ethnicity", "match_score", "match_method"]
]

,Title,Wiki Title,Release Year,Origin/Ethnicity,match_score,match_method
1732,U.S. Marshalls (1998),U.S. Marshals,1998.0,American,96.000000,fuzzy
1840,"X-Files: Fight the Future, The (1998)",The X-Files,1998.0,American,100.000000,subtitle
1907,Friday the 13th Part 3: 3D (1982),Friday the 13th Part III,1982.0,American,91.891892,fuzzy
2038,Halloween: H20 (1998),Halloween H20: 20 Years Later,1998.0,American,100.000000,subtitle
2056,Ever After: A Cinderella Story (1998),Ever After,1998.0,American,100.000000,subtitle
2104,"Navigator: A Mediaeval Odyssey, The (1988)",The Navigator: A Medieval Odyssey,1988.0,Australian,98.245614,fuzzy
2586,Howling II: Your Sister Is a Werewolf (1985),Howling II,1985.0,American,100.000000,subtitle
2679,Allan Quartermain and the Lost City of Gold (1...,Allan Quatermain and the Lost City of Gold,1987.0,American,98.823529,fuzzy
2973,Meatballs III (1987),Meatballs III: Summer Job,1987.0,American,100.000000,subtitle
2995,Poison Ivy: New Seduction (1997),Poison Ivy: The New Seduction,1997.0,American,92.307692,fuzzy


In [53]:
# unmatched movies (mostly films the Kaggle dataset doesn't contain, e.g. French/Italian/German titles)
movies[movies["Plot"].isna()][["MovieID", "Title", "Genres"]]

,MovieID,Title,Genres
28,29,"City of Lost Children, The (1995)",Adventure|Sci-Fi
36,37,Across the Sea of Time (1995),Documentary
45,46,How to Make an American Quilt (1995),Drama|Romance
50,51,Guardian Angel (1994),Action|Drama|Thriller
52,53,Lamerica (1994),Drama
...,...,...,...
3849,3919,Hellraiser III: Hell on Earth (1992),Horror
3850,3920,"Faraway, So Close (In Weiter Ferne, So Nah!) (...",Drama|Fantasy
3865,3935,Kronos (1973),Horror
3874,3944,Bootmen (2000),Comedy|Drama


In [74]:
out_path = folder / "movies_wiki.csv"
movies.to_csv(out_path, index=False)
print(f"saved {movies.shape} -> {out_path}")

saved (3883, 14) -> dataset/ml-1m/movies_wiki.csv
